# Module 12 · Solutions

In [ ]:
import pandas as pd, numpy as np
from scipy.optimize import minimize
import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
uni = pd.read_csv(BASE+"nse_stock_universe.csv", parse_dates=["date"])
px = uni.pivot(index="date", columns="ticker", values="close").sort_index()
px.loc[:"2024-09-01", "TATAMOTORS.NS"] = px.loc[:"2024-09-01", "TATAMOTORS.NS"] / 5
rets = px.pct_change().dropna()
n = rets.shape[1]
sectors = uni.drop_duplicates("ticker").set_index("ticker")["sector"]

## 12A

In [ ]:
# Ex1 - the free-lunch counter
cov_d = rets.cov()*252
same = ["HDFCBANK.NS","ICICIBANK.NS","KOTAKBANK.NS","TCS.NS","INFY.NS"]           # 2 sectors, correlated
spread = ["HDFCBANK.NS","SUNPHARMA.NS","ITC.NS","POWERGRID.NS","MARUTI.NS"]        # 5 sectors
for name, basket in [("clustered (fin+IT)", same), ("sector-spread", spread)]:
    w = np.ones(5)/5
    v = np.sqrt(w @ cov_d.loc[basket, basket].values @ w)
    avg_single = np.sqrt(np.diag(cov_d.loc[basket, basket])).mean()
    print(f"{name:<20} portfolio vol {v:.1%}  (avg single-stock {avg_single:.1%})")
print("Same count of stocks; the sector-spread basket is meaningfully calmer. The heatmap's blocks, cashed in.")

# Ex2 - constraint humility
mu = (rets.mean()*252).values; cov = (rets.cov()*252).values
def max_sharpe(cap):
    cons = [{"type":"eq","fun":lambda w: w.sum()-1}]
    r = minimize(lambda w: -(w@mu)/np.sqrt(w@cov@w), np.ones(n)/n, bounds=[(0,cap)]*n, constraints=cons, method="SLSQP")
    return r.x
for cap in [1.0, 0.10]:
    w = max_sharpe(cap)
    print(f"cap {cap:.0%}: Sharpe {(w@mu)/np.sqrt(w@cov@w):.2f}, holdings>1%: {(w>0.01).sum()}")
print("The cap 'costs' a slice of (in-sample!) Sharpe and buys a portfolio a human can defend after a bad")
print("quarter: 12-15 names with reasons, vs 4 concentrated bets on estimation noise. Easy client choice.")

## 12B

In [ ]:
# Ex1 - the turnover tax (quarterly re-optimisation in the OOS period)
split = "2023-12-31"; r1, r2 = rets[:split], rets[split:]
def mv_w(cov_m, cap=1.0):
    cons=[{"type":"eq","fun":lambda w: w.sum()-1}]
    return minimize(lambda w: w@cov_m@w, np.ones(n)/n, bounds=[(0,cap)]*n, constraints=cons, method="SLSQP").x
def ms_w(mu_v, cov_m, cap=1.0):
    cons=[{"type":"eq","fun":lambda w: w.sum()-1}]
    return minimize(lambda w: -(w@mu_v)/np.sqrt(w@cov_m@w), np.ones(n)/n, bounds=[(0,cap)]*n, constraints=cons, method="SLSQP").x

quarters = pd.period_range("2024Q1", "2025Q4", freq="Q")
strategies = {"max-Sharpe": lambda hist: ms_w((hist.mean()*252).values, (hist.cov()*252).values),
              "capped 10%": lambda hist: ms_w((hist.mean()*252).values, (hist.cov()*252).values, 0.10),
              "min-var":    lambda hist: mv_w((hist.cov()*252).values),
              "1/N":        lambda hist: np.ones(n)/n}
for name, fn in strategies.items():
    w_prev, pnl = None, []
    for q in quarters:
        hist = rets[:str(q.start_time - pd.Timedelta(days=1))]
        w = fn(hist)
        cost = 0 if w_prev is None else np.abs(w - w_prev).sum()/2 * 0.0015
        qr = rets[str(q.start_time):str(q.end_time)] @ w
        pnl.append((1+qr).prod() - 1 - cost)
        w_prev = w
    total = np.prod([1+p for p in pnl]) - 1
    print(f"{name:<12} OOS 2yr return net of quarterly turnover costs: {total:+.1%}")
print("\nmax-Sharpe pays the biggest tax: noisy inputs change every quarter, so IT re-shuffles hardest")
print("(the drama queen billed by 11B's toll). 1/N pays ~nothing - it never changes its mind.")

In [ ]:
# Ex2 - homemade shrinkage
mu1, cov1 = (r1.mean()*252).values, (r1.cov()*252).values
def oos_sharpe(w):
    pr = r2 @ w
    return (pr.mean()*252)/(pr.std()*np.sqrt(252))
raw = ms_w(mu1, cov1)
mu_shrunk = 0.3*mu1 + 0.7*mu1.mean()
shr = ms_w(mu_shrunk, cov1)
print(f"OOS Sharpe: raw max-Sharpe {oos_sharpe(raw):.2f} vs shrunk {oos_sharpe(shr):.2f} vs 1/N {oos_sharpe(np.ones(n)/n):.2f}")
print("Pulling every estimate toward the average dampens the noise the optimiser feasts on - shrunk")
print("typically lands between raw and 1/N. You've built the intuition behind Ledoit-Wolf/Black-Litterman:")
print("don't trust your estimates fully; blend them with something humble.")

## Ex3 — the client letter (model answer)

*"We hold about fifteen companies across different industries, with no single one allowed to dominate. A computer can always find a mix that would have looked slightly better in the recent past — but 'the recent past' is a small sample of luck, and portfolios built to fit it tend to disappoint when conditions change. Ours is built instead on the one thing that reliably persists: companies in different businesses don't stumble at the same time. That costs us a little sparkle in good quarters and spares us concentration losses in bad ones. We think that's the right trade for money that matters."*

*(Five sentences, no jargon, and every clause maps to a result you produced: overfitting to estimation noise, the persistence of covariance vs returns, the cap's insurance trade. If you can write this letter, 12.5 will be a comfortable module.)*